In [1]:
%pip install -q ultralytics==8.4.203 kagglehub pyyaml


In [2]:
print("Iniciando a preparação...", flush=True)
from pathlib import Path
from collections import Counter, defaultdict
import json, hashlib, random, os, time, shutil, gzip, base64
import numpy as np
import cv2
from PIL import Image
import kagglehub
import yaml
cv2.setNumThreads(1)

# Lista compactada das mesmas 3.500 imagens e dos mesmos grupos.
# Evita uma linha gigantesca no editor do Colab.
DADOS_COMPACTADOS = """H4sIAAAAAAACA5Xdu47kOruY4Xv5401D5HciQ0eGA1+BYQx03Mm24cC+f6vWdLHa06JeTbJWMA/U0qtqkVRXqf77
f//XNv+fOf/r3/71n3/91//2X36VqUiesv7KXrzIrzz9p//9v/79V93KutWmafftSDrnNbXSamq6rJMvsh22vOC/
/m36H/+G26z/bLPIUScJT8uyeNI6bWmO3NIixxal7L7L+tfbXPLuh2o6pnZuU+eS6uGStjkvodLaNl/t5/jIn8D8
FOpTaE9hvYDz//33/7n/r/+zb7+mXxP8e4Z/L/DvAv+u8O8G/+7w73H77xmOP8PxZzj+DMef4fgzHH+G489w/AWO
v8DxFzj+Asdf4PgLHH+B4y9w/ALHL3D8AscvcPwCxy9w/ALHL3D8CsevcPwKx69w/ArHr3D8Csevl8f//w0Qdl6e
p2Lt90/6FefINR+xp6PNS9JNJbXz39PWFm1ibd+0/d5kvt+kTOU9NhaftrLtLUV5jY1Z1jTL4anNs+6t6DmobX+7
zbnMLY5z77bse9LWIjWfNS01T7bsHmV5vp//nKRzUNllnuec5J9N5i2nWtqe5mXWNewc090eb/L3aKu219haS/s5
ep811zUt8zGdO92mfVva0XT9y03WiHW30OQmknQ5t1a1Huf52nSra1tL+ONN/pPoV8tbdTm3UTXO83NWOOcu5zmX
WvPm87S3Of52k3EsRc6CWff13Mu2nNOMHGm3YvOUi3grF5u8el1mewjL9BDqQ/f0B9eHrj1wn5f4AygPXX3ornbw
22DaZ0tjkAkUAkJACRgBJxD3IFOHTB0ydcjUIVOHTB0ydcjUoVCHQh0KdSjUoVCHQh0KdSjUQaiDUAehDkIdhDoI
dRDqINRBqYNSB6UOSh2UOih1UOqg0MEgg0EFgwgGDQwSGBQwCGBw/A7H73D8DsfvcPwOx+9w/A7H73D8AccfcPwB
xx9w/AHHH3D8AccfcPwVjr/C8Vc4/grHX+H4Kxx/heOvcPwNjr/B8Tc4/gbH3+D4Gxx/g+Nvvx6uBvqNul++rqto
qUmnsKRhc5r3vaRj3Txv4duSv+46Ft5m/F6oSpuOqdm5llI9V2s6p0XLmrZtNrFtXw5bLzZ5uZP6FNpDGBfu2wjS
R4gxyAQKASGgBIyAE4AOTh2cOjh1cOrg1MGpg1MHpw5BHYI6BHUI6hDUIahDUIe47HDxS27n//7Z1q+lLc2mraV1
ms+rR63nb/mxHGnTvZrorNP8VUYebPP3lWPNvqy7TmluuSS1uaUWbUled1lkyrXs+nCb51BWft+Oknk/1nMjEXJe
jeZpSuem9tSauxxa9+mQv93mWibbtqmmpSznRbOppMVmSUeTqS7LUfZ9+9tt7ntdj72ch13Oa6XKtJ49ty1F9uO8
Pi+e39c4ub3GvWrKU/h4i/4UxkOoD1175N4VH8CrQ/72m9CnSGOQCRQCQkAJ4FE4gbgHjTo06tCoQ6MOjTo06tCo
Q4MOZYIOZYIOZYIOZYIOZYIOZYIOZYIOZaIOmTpk6pCpQ6YOmTpk6pCpQ6YOhToU6lCoQ6EOhToU6lCoQ6EOQh2E
Ogh1EOog1EGog1AHoQ5KHZQ6KHVQ6qDUQamDUgelDkYdjDoYdTDqYNTBqINRB7vs8HNip/L+e5u1fQ3Lc9rWyOcC
+zX1LMuWjuWc7u2yalv335tU2uTXr9Sv2XKej01THF5ff7tcUl3mOdVW97Ie8zK7Xmzyah9zewjLQ/dse/3igK48
dHLhvp23vp4dg0ygEMB9UAJGwAnEPQjqENQhqENQh6AOQR2COgR1qNShUodKHSp1qNShUodKHSp1aNShUYdGHRp1
aNShUYdGHRp0kAk6yAQdZIIOMkEHmaCDTNBBJuggE3XI1CFTh0wdMnXI1CFTh0wd8mWHizHX4muIPP8379t06Dl6
19dNmn/e7WvrOY7H4vtc5nn9ulFut2PVa4vTU+hPYbuA3463D39jkAkUAkJACRgBJxD3IKhDUIegDkEdgjoEdQjq
ENShUodKHSp1qNShUodKHSp1qJcdfv6a2+s2/tcfr9ZzPh4a6Si1JV3q/noXvqVS8nzOr5fi61cZv/ultM+bV9DV
R66/MtnFU9gewvLQ+UN3tYPfTluflIxBJlAICAElYAScwH0HnaCDTtBBJ+igE3TQCTroBB10gg46UYdMHTJ1yNQh
U4dMHTJ1yNQhU4dCHQp1KNShUIdCHQp1KNShUAehDkIdhDoIdRDqINRBqINQB6UOSh2UOih1UOqg1EGpg1IHow5G
HYw6GHUw6mDUwaiDUQenDk4dnDo4dXDq4NTBqYNfdriY1fV3JGVVmcoaad9eH3PZSk61LSUV122q+6zSvv5eH083
ucjU8jRZspzXpNEiLVtossnmzfI8Z9kvNnm1wfd8DeGV+9alL27GIBMoBISAEjACTgA6VOpQqUOlDpU6VOpQqUOl
DvWyw8Vr9n35+3VoaxJaUivHfP4aHGuqIjW9PlWmUtq2ta/XbOVN/v7MkxSfdRZPx7advwY1z6n6sqRpzdu8hqzT
+64IbtK/3rfyS4992fYqybb6+oTS3tKyz2sSX5d9qy3Ctr/dpq/nFqY4UtbVX3+FWdMyLa+32Pha8rHJGeLxNn9v
cp6XeRXPaT/Ws6bYkWZzSeVY9v2QbdcjP9xkvHezHCUXb3v65zTpfu7mfOZIMR3nBaC65/J8m1+fSmsuMvmUtnXL
50n3/bxQzVPKq1s59iOv7+vPg022r7c/yVZVl+RzmV8n/RWzTekwPVrObWrH1UkfvjCfuOkplIdQH7p46Ooj5093
sA+06NojF09/8Of+AcLy0D39wfWhuzribxfGfvtgDDKBQkAIKAEj4ATiFtgEHWyCDjZBB5ugg03QwSboYBN0sIk6
ZOqQqUOmDpk6ZOqQqUOmDpk6FOpQqEOhDoU6FOpQqEOhDoU6CHUQ6iDUQaiDUAehDkIdhDoodVDqoNRBqYNSB6UO
Sh2UOhh1MOpg1MGog1EHow5GHYw6OHVw6uDUwamDUwenDk4dnDoEdQjqENQhqENQh6AOQR2COlTqUKlDpQ6VOlTq
UKlDpQ6VOtB80mg+aTSfNJpPGs0njeaTRvNJo/mk03zSaT7pNJ90mk86zSed5pNO80mn+aTTfNJpPuk0n3SaTzrN
J53mk07zSaf5pNN80mk+6TSfdJpPOs0nneaTTvNJp/mk03zSaT7pNJ90mk86zSed5pNO80m/nk/+vE/k+XMnb11K
1XVKIcfrE8Y5p1oPT2VaZXU9LNav3W4Ptqm/7zj6sccUJZVja0llk9Ra3lNddCkmEarxt9t0m6d5sS3F0krSGtu5
OckpxyTlCJ32Up9v8+vO6Cr7suSWqjdLqsuUln2RpK2sEpvZdujfbrNN8+6HeLI4A5xbmlKTSdJ0xN7WZZ3LZH+7
zXMnN9tN0iFtS68HVaX59VHwWPMx1TWmWreH24x3zzVL2WSJVNqkSePcXN2PkvLreVaLzNtkx/Nt/v6Dy+ZzTLrN
55aqn/tp5fWUsjPAtEz7vkTsdXq8zd/3MbfcpO6uaS3H+VI6NNKi+/kTsu2Lz+t58HKxycuY01P4eIv6FNpTWB/C
Zy6eHks8PZbPrUeGT/fxqWsX7tvFr8+FxyATKASEgBIwAk4g7kGjDo06NOrQqEOjDo06NOrQoENM0CEm6BATdIgJ
OsQEHWKCDjFBh5ioQ6YOmTpk6pCpQ6YOmTpk6pCpQ6EOhToU6lCoQ6EOhToU6lCog1AHoQ5CHYQ6CHUQ6iDUQaiD
UgelDkodlDoodVDqoNRBqYNRB6MORh2MOhh1MOpg1MGog1MHpw5OHZw6OHVw6uDUwalDUIegDkEdgjoEdQjqENQh
qAPNJ4Pmk0HzyaD5ZNB8Mmg+GTSfDJpPBs0ng+aTQfPJoPlk0HwyaD4ZNJ+M6/nkxdr0/RG+X9u5Js22Lmnd/Vzu
etTUYlnT7vNWdqtHsa/3auWJtvk1av/asy5tej1B3M+Fsx7Hnlp9PU261XLYVmre9GqbV3v5/uYIlv4Mlr6cZPl4
k3EFvz0v8PPc5LHIKAoKQaEoDIWjoB4Ze2TskbFHxh4Ze2TskbFHxh4FexTsUbBHwR4FexTsUbAH/74I9hDsIdhD
sIdgD8Eegj0Eeyj2UOyh2EOxh2IPxR6KPfS6x88hJ97v+/o1V5nzUTRJHJbUliPNr7u20qZaiu51Wt4Fb5+B/9pi
Hx9Y6mMZT+XjTbYr+K3i5yG8Y5FRFBSCQlEYCkcRIBr2aNijYY+GPRr2aNijYY9GPdpEPdpEPdpEPdpEPdpEPdpE
PdpEPdqEPTL2yNgjY4+MPTL2yNgjY4+MPQr2KNijYI+CPQr2KNijYI9y3ePnmFOn3B9N68thS/Zz0Mn6ej7Wlpp7
SzGJr8eaS7P3jhXe6O9tTk1lOtcUaXVdz222KS215lRqa3Xzc6kmx9U2L3ezPJb+WLan8nKT34p/5nhjkVEUFIJC
URgKPtoAodhDsYdiD8Ueij0Ueyj2UOxh2MOwh2EPwx6GPQx7GPYw7OHYw7GHYw/HHo49HHs49vDrHheXztdDA7++
ZW72dd7WSHq8vrnu9dDwpZ4X0CLnVdNk8fJ+O0iWxxutk+g06f76poItqTdNbZOWprqeP06npfh0tdGrTfaVBcpy
uc1vfT6z+7HIKAoKQaEoDIWjCBANezTs0bBHwx4NezTs0bBHu+5x8UJ+PatDv17Iuk6raZrt9XHXednS0iLSYef8
wtdorX+voz7e6Kqb1bmd8xOZ7PURzZaaHiXtdkwSi4m7/MVGf79VzHIJnVpJ+djPX7myLun8VVvSokfU3aajLP58
o+8P5nqd4pxXHUeZkq62pbq6Jd+ms4Lvq9n8fJtf7+uy4m0+NBV9vf1sPad+c/ZIbY51yfuq+v4agXz7XNFPz0ey
PpXyFMZT+PhHX76Svn/D3mf1eUMyk8JEmCgTY+JMgkjmLpm7ZO6SuUvmLpm7ZO6SuUvhLoW7FO5SuEvhLoW7FO5S
uItwF+Euwl2Euwh3Ee4i3EW4i3IX5S7KXZS7KHdR7qLcRbmLcRfjLsZdjLsYdzHuYtzFrrtcjO3W36u/+lyOtra0
HeZJ99f7/2vez4lObOu2LEte31/cfPvY1M8m/5Tf9/CzuLshmUlhIkwe7K4xcSZxRa5OS+tfidRcfKvnPC63es4N
j0hzyZJ20diK+xTzu6Q/3+i8q6yv53BWP6fG56Q2LXL+x9Y6Le57PVSfb1S/FqPrtNQ9JzM7krZpSm3ZlxSaY93m
pW7r5Y5e7uZzWR5LeSr1KYynsF3B76+Mzxdo3pDMpDARJsrEmDiTIFK5S+UulbtU7lK5S+UulbtU7tK4S+Mujbs0
7tK4S+Mujbs07PLt+9pvSGZSmAgTZWJMnAl2ydwlc5fMXTJ3ydwlc5fMXTJ3KdylcJfCXQp3KdylcJfCXcp1l5+D
+zmGf935qZPkstk5OdSWzwlD2VKNImleXazZ6wbx+/hun7b42mKfHKJsV/D7kXwWgzckMylMhIkyMSbOJIgod1Hu
otxFuYtyF+Uuyl30usvFy/p18L8nwudUdc/7ud7JuerrqfTnRHh9PXDR2qznK9xbfd8CvH162meTj6Reye/H8lm1
3pDMpDARJg9215g4kyDi3MW5i3MX5y7OXZy7OHfx6y4XL+z+J7dfx3zOr1vxdLy+c0GbTWmJmM91fTns9S3AVd8/
t/FG29fzQ4tEc0/rsu6vT8lrqj7Pqe1Nm0+ivl5uc7ybT2S7gt/zfNZONyQzKUyEiTIxJs4kiFTuUrlL5S6Vu1Tu
UrlL5S71usvFy7q/nfXXoXL+Nkx7ytv+eltTWVK1taRp1mPW84p92NcgUG4/GNG+vUP2gbTH0q/k96Puy8E7kpkU
JsJEmRiTBwcdQD5fMnxHMpPCRJgoE2PiTLBL5i6Zu2TukrlL5i6Zu2Tukq+7/Lg05Om1SPn9Zpg8LVJ0OYfRpZ2X
hnPUTHWtr6dyH5LXTcu8fd0UL/nxRmXaljlLTfkISTqd/5l1r+fMM+dtknPo1vIXG7XfM1lt+yqvP/U3rem8orVz
hWZ7knKuzo5J3N/vdih377T/tp+PZH4s9bG0p9Kv4PdT3pf3dyQzKUyEiTIxJg8OOogIdxHuItxFuItwF+Euwl2E
uyh3Ue6i3EW5i3IX5S7KXZS7GHcx7mLcxbiLcRfjLsZdjLs4d3Hu4tzFuYtzF+cuzl38usvP0Snr+/ldv9Zp333b
WrKSl9eXJUhqyzk6vb5xeyuWFyvvn3v3Pvtvm3wi4wp+OxT5XAdvSGZSmAgTZWJMnAl2Ue6i3EW5i3IX5S7KXZS7
6HWXq9d16/PDbd192s4JXJ1f88Ppn2+1nFOb7dg3s+z+/rly/2r9/NH7gZQr+f1YPhfsG5KZFCYP9kWZGBNnEkSc
uzh3ce7i3MW5i3MX5y5+3eXnC7u83mTw+8Z4LPb6DtUpldcNP9Vc0nxMc5qmtUUuu0q8D1AfbPT3TcmWbZnaLqm+
PjWmx55TNdPU2h6rZtcpz1cbHe/nIymPZbuS31J+vjn0jmQmhYkwUSbGxJkEkcxdMnfJ3CVzl8xdMnfJ3CVzl8Jd
Cncp3KVwl8JdCncp3KVcd/l5aZD3pOfXvFub11jTFhZJfdlS27ctHft5vZg8T8f7CTrl7s2Hny0+ge0Kfj+QzzTo
hmQmhYkwUSbGxJkEEeMuxl2Muxh3Me5i3MW4i113uXhVv74H8fdfzGyd8r6tktb8ev61HpqWo0jaNUeUfS8a70z+
eKMx2X7+gtR0rpxq0rZZmvOa06oi61GWJb8/9Vf89lfgtcn8WLanMq7g95KfSdgNyUwKE2GiTIyJM8EuwV2CuwR3
Ce4S3CW4S3CX4C6Vu1TuUrlL5S6Vu1TuUrnL4Pfo55VBS7/LMq/y+vDu6w+Z8+u7c3VKdZpbml2r+LLWPH89cr7E
440uupV1K5HW9s9GQ9IibU/HPO2x76Heb+3/xUbt9enB+Xh9aPn1IPuaz+VFjnPxHLLa1FSX/HSj7X1hlMPPfbEp
NTuvsecKxVObWiQ5d1X8yLq8v23gLzbqonlvJSf3Kf/zvqDzEn6u9c8Jy5qnuZbtiKuNjg/+kdTH0p7J9rmCj95B
9fmCxzuSmRQmwkSZGBNnEkA+X/h4RzKTwkSYKBNj4kywS+Yumbtk7pK5S+Yumbtk7pK5S+EuhbsU7lK4S+EuhbsU
7lKuu/y8uFru71KrSzQv5wh1rvUs6T6XVNthaZ9fA0ttUeS90fp4o0fNRab6+hqTeH1BjEmazyt4imVfjzpNyy7L
1UbHm3wiL/fze5/P3xpuSGZSmAgTZWJMnAl2Ue6i3EW5i3IX5S7KXZS76K+nr+vyvi35azriONeROZXX3VhdlyXN
W9a0Tm2xPMk5K3vP7trtq/W1yemxLI9lu5Lfj/pz3+CGZCaFiTBRJsbEmQQR5y7OXZy7OHdx7uLcxbmLc5fgLsFd
grsEdwnuEtwluEtcd7m4NFj/a83sZW2lSSr19Y6yw85xdF4s5WVZjlJff6z5epuaTLzR39sscS4l2/r6XjB/PVwq
9nMxWaa05DnLXI+5efuLbX49sWqVVdq8p/XYa9JtORen+3QO0C2345iOth/laqOXxz49ls+3qU/l402Wx7t5+bO/
vzT6jZA7kpkUJsLkwe4aE2cSRBp3adylcZfGXRp3adylcZeGXT5fqHpHMpPCRJgoE2PiTLBL4S6FuxTuUrhL4S6F
uxTuUriLcBfhLsJdhLsIdxHuItxFrrv8HMb89R7H38PYsfjRSkRat2ivp6+fo+Tr0SlNo9ikbcvvL7iU/GCjvwfc
Kofv7RwMq+bjHMS3SO3Y1uQ6n5P9rc5y5L/e6DybqrYlFXl9wFFyPcfaY0mtWV1s2WLpA+6Djf5+XN0217lN58pj
mV9PiFzOpfuyH+eOrnv2Xaf5/DFX2xwHfSTLY6lPZb2C318cfcVyRzKTwkSYKBNj4kyCiHMX5y7OXZy7OHdx7uLc
xblLcJfgLsFdgrsEdwnuEtwluEvlLpW7VO5SuUvlLpW7VO5Sr7tcXVr7U7jKuc7a17qkvLm9bvKc11bJS5KcI0+T
6r6/G5QHG7Wvr/Z2O8p51a/b63FP5u0crmRN81HWsuh0XrzlaqM/N6n9jijJz1+7Hkh9LO1Kfo/ebyfekcykMBEm
yuTBETmTINK4S+Mujbs07tK4S+Mujbs07PL5wuE7kpkUJsJEmRgTZ4JdMnfJ3CVzl8xdMnfJ3CVzl3zd5eLi+nki
dMybtvm8mh4xRVI9r93njHhPWnydWq7Lrl9vahbhjX49aFltKsd+LgD0XFVoqXOaPaa0mbYi8yGh29U2x7v5RLYr
+D3PZ918QzKTwkSYKBNj4kyCiHAX4S7CXYS7CHcR7iLcRa67/HxZx+u+6u+JyDTt58p4l7Tq+eJWW6fU1rWm0ta8
rstepuPryzfl9s36n03+Kb/v4WeRckMyk8JEmCiTB0fkTOKK/Dwt/3w/w+/T4q1MWkpL67FNSScrad7E01ZLm7ay
LtIvYfZ4o/PcVOqxpWV63SMpc7zejOZpjnMnj+y77OvVRseb/FN+P+zPwuuGZCaFiTBRJg+OyJnEw9PyemvB+5aQ
1uP8PTty0jjOYep1c2jec0lSt2kLabHo1/uHxf9is/vhbT1Uk5RFzt9tzamu55m3Oa/afN7ORcjVZu82+qf9fvCf
JeUNyUwe/CBhokyMiTOJZ6em5X5ipvPELOd84rzUbq83oS/T63bdnPbXHw3ztB2yvhvE440uRz68xLk6zdXPsz3v
qZVlTpus2Uy3xfVyo+NNPpJyJb8H+ixzbkhmUpg82BdlYkycSQD5fCHzHclMChNhokyMiTOJhy/s90fyfp2/EOV4
fbvdkYu9hsJ4PUZlSrtNOebjvGLa+8feviWsfT7kJ6N3AX2+BfmOZCYPfpAwUSbGxJnEFbk6Ke3r8ym/6rrvMZul
w/9561PbUmvlSEcpscR+XnGW9+Kl3Z+V1j/y8kD6lfx+LJ+J/w3JTAoTYaJMjMmDgw4iyl2Uuyh3Ue6i3EW5i3IX
ve5y8cK2/H6s7rxve3l9KmzSOemse5qbabLaXt/Y5PsRX9vU23eP9C3+Cb/vX/9b1h3JTAoTYaJMjIkzuQz446SU
kvuHU2QrdZ/L6xu1/5kwvR4K+/pK0nmNEJnOS/t7wqR3f2H8tsk/5fc97PPYO5KZFCbCRJkYE2cSV+TnadHcP4lz
LFPdtihpaus55RRbUpW2purLmstay3m2vzZ693eE35t8LuOxbFfy21F/vnH5jmQmhYkwUSbGxJkEkcJdCncp3KVw
l8JdCncp3KVwF+Euwl2Euwh3Ee4i3EW4i1x3ubg0vD5c9Pte03ZOAffZS1p0y0k3LWmueU35EInXBxWrvTcqt7/G
r01Oj6Vdye/H0udBdyQzKUyEiTJ5cETOJIgYdzHuYtzFuItxF+Muxl3susvPF7ZJvBc++TiqR3ndIX89MafNJbWY
l7SHLLaYZTu+3sOkdzfMv23ykWxX8vux9FvrdyQzKUyEiTIxJs4kiAR3Ce4S3CW4S3CX4C7BXeK6y8UL+/Vch98v
7CXPJWLVJMt2rnyanCv6c2qdlpKXMs/e/Pj6C6fa7cu1b/KR1KcyruD3g+43Gu9IZlKYCBNlYkycCXQpn++5vSOZ
SWEiTJSJMXEm2CVzl8xdMnfJ3CVzl8xdMnfJ111+Xhm8P4vm1zm0zVtb/LweyJHOhdWW6rGeU7uY53Ur+yrz++fe
/R3p2yb/lN/38LP0uCEPtlKYCBNlYkycSVyRn6clpv6kS5nXVnSZUzY9Xm/6lrSYz0l2DZ9sOmp7/9y7P+N82+Qj
ma/k92P5TLFvyIOtFCbCRJkYE2cSRIy7GHcx7mLcxbiLcRfjLnbd5eKF/Vq+6ftL7Y/59VCuJXx7fUJiSlU0v74a
NXuV0GP6evOD1tuXa9/kn/L7Hn7u9t2QzKQwESYPdteYOJO4Ij9PS3s9Vvj3MDCdo+ouqikv6klj19eSfk5WtbTN
t1jfD/PRuz/kfNvkn/L7Hn5mczfkwVYKE2GiTIyJM4kHBSd9TZy/DswmknEFPz9V+oB7IzKKgkJQKApD4Sioh2AP
wR6CPQR7CPYQ7CHYQ657/HwhfYZ+yyTjCn77oX2KcCMyioJCUCgKQ+EoqIdhD8Mehj0Mexj2MOxh2MOue/z5QvLX
dOh91AWlPpb2VL4fFGCDW+Ofj7bciIyioBAUisJQOIoAYdjDsIdhD8Mehj0Mexj2MOzh2MOxh2MPxx6OPRx7OPZw
7BHYI7BHYI/AHoE9AnsE9ojrHn9eaWL6fJTaBn+r+LxV5EZkFPxTBIWiMBSOIq7Ej3Ll8zw4G9wMr5/fwbHIKAoK
3g9FYSgcRVyJH+WkrwBtcK+wfn5ZxyKjKCgEhaIwFI7issef4errbSXv/Rnczvv8zelGZBQFhaDgPTUUjiKuxI+v
B/x8rZmN3h9cPuVuSGZSmAgTZWJMnMlllx8BX4/w//o7sI3ezFv67aw7kpkUJsJEmRgTZxJX5Mc3NX17n5mN3jVa
+p2nO5KZFCbCRJk8OCJnElfkR8HyuaXqo7d4Sv+L4x3JTAoTYfJgd42JM4kr8qOgfJ5p5xllvZLff2z/K+YdyUwK
E2GiTIyJMwkihbsU7lK4S+EuhbsU7lK4S7nu8uOVZZ8Pbnkh+RjWK/h99/rofkcyk8JEmCgTY+JMgkjlLpW7VO5S
uUvlLpW7VO5SuUvjLo27NO7SuEvjLo27NO7Srrv8+F6f6fMNQD56h6P2v2XckcykMBEmysSYOJO4Ij8eWVA+l6zh
h/f7Hd47kpkUJsJEmRgTZxJX5EdA8fdnnHz4ifh+4+qOZCaFiTBRJsbEmcQV+fEhse+z6dEnzOtnhndDMpPCRJg8
2F1j4kziivz5XoLpHEbeL8Hhh477HcA7kpkUJsJEmRgTZxJX5Mcn4qZvU8HhJ4Q/E7cbkpkUJg/2RZkYE2cSV+RH
wc9XR/jwk6ifmcwNyUwKE2GiTIyJM7ns8mdAeT1G4uuXOEbL69bvKdyRzKQwESbKxJg4k7giPz8h0qfkMVobtr7I
uCOZSWEiTJTJgyNyJnFFfr6x9fPMkyijd2f1yfQdyUwe/CBhokyMiTOJK/Kj4Of7sUMIxhX8/kP7iH1HMpPCRJgo
E2PiTLBLcJfgLsFdgrsEdwnuEtwlqEv2z+CgY5KZFCbCRJkYE2cSf5Ly/5F3E7v+Z7n/Z73/57j/5+k//uML+DXI
BJSAEXACQaDiUXz9c1z/c7n/Z7v/Z7//53r/z+3+n9+f+Ig6+PcM/17g3wX+XeHfDf69wr9P7wNsI5AJFAJCQAk4
gSDwPs46DUVBISgURaCoKBqKr5NW81AYCifx/uWoZSgyCkERKN5nToZCURgKR1FRNBLvN9NWHQpBESgqiobifW5t
KAoKRWEonIS8m/pQZBQFhaJwFIHiffaHA5YYioqikdB30+HA8n6O5o1wFBVFQ/E+t+PhR1AoCkMRJL420YaX9Uqg
EbCv09aGF3UrKASFoXAUgaKiaCi+XhxtOLiYkvB30+FF3TMKReEoAkVF0VC8Xx/DgcEFhZGId9PhRT0MhaOoKN5n
bnhBjoJCUCiKQNFI1HfT4UW9CgpFYSgcRUPxPi/Dy34tKAJFJdHeTYcX9ZZRFBSGwlFUFA3F1+vjXBwMiTIJIu+F
xTlhHglHUVE0EgX34z0JvhGCQlEYCuxRuMfUD7cMSWZSmCgTY+JMgkk/PzIklUlD0suNBr1cUAgKRWEogkTBPS28
jYb78bkM2JAoE2cSTCqTxqS/CHxIChNhYkg+F7cYksJEmTiTxqSnG96K/Vxqx8SYBJOKRHrdNiTCJJi8u+ThaPq+
d3FHlIkxcSaVSUPyvvtwvtCHRJgYk8qkn4DhGKOFiTJxJsGkIbFedziYmTBRJs6k1x0PaIWJMQkmlUlD4r3ucJhw
Y9K7DC/wXpgIE2XiTIJJZdKQRK87HEkiMxEmysSYOJPKpDHpL4bhkBWBpPa6w/GoZiaFiTMJJo3J+0yX4WDzvudx
R4xJRfK+mXDOdYYkMylMjIkzqUx63eFg028Y3JBg0oiUvrotMiSZSTCpTBqTfhqHK6i+dL0hysSYOJK+Vis2JJmJ
M2lMejofEmGiTIxJMKlI+lqtxJBkJsLEmASTfgKGb4vp68Yb4kwqk4akr9VKG5LCRJkYE2cSTCqTxuT9kpJpSARJ
X6vJ+AZoYSJMjIkzCSaNSU83HI/6iu+GVCR9rSbDkaSvj26IMnEm/aCHw0RfN94QYxJMKpOGpK/VZDjY9IXYDVEm
xiSY9DM9HEn6iu+GOJPKpCHpazUZjiR9rXZDChNjEkwqk8akn4DheNSXljfEkfS1mgwv8H0hdkOEiTJ5n0YdDhN9
CXVDnEkwqUwakr5W0+EY0NdqN6QwaUze50iHw0RfZd0QY+JMgkkl0idkOhpr+nxsLASFojAUjiJQVBQNRV/dqg5J
YWJMnEkwqUwak/4bZkMiTBRJ/zk+FL1/DElmYkyCSWPSXwvDDyb0JfINUSbOpCLp619tQ1KYGJPKpDF5n2mbhkSY
KBNnEkj65dbykCgTZxJMKpPGpJ+A4RDTR5AbIkwMSV//2nggykyCSWXSmPQuw8tuX4vfEGViTBxJX//a8OLdV4s3
pDJpTPppHA4TfS1+Q4SJMjEmjqSvf204kvT17w0pTISJMjEm/aCHg01faN+QyqQh6etfG44kff17QwoTYaJMnEll
8j5HPhyP+lr8hjQkff3rw2GiL0tvSGGiTJxJMKlMGpP3i8GHQ1Zfi49JX//6eNFSmAgTZWJMGpP+YhgONn39e0OC
SSXSpww+Gmv6EDwWisJRNBKGe9oHxbHAYzE8FsNj6bOAscCjdTxaxz3tQ8dY4H5oX+27D0lmUpgIE2XiTIJJY9Kv
BDEkFUkvN/xQfkFhKAJFJWG4H4b7YfhTHHs47kef340F7mmfuo0FN+33abwNSWYiTIxJY/IuG9OQKBNnEkwqkn6f
JvKQFCbCxJm8T2OUIVEmxiSYVCYNSb9PEzIkmYkwMSbOJJj0F8NwKtTvPN2QyqQh6ZOuGE+pMhNhYkyCSU83HA37
FPCGOJPKpCHpE8Xhk3a0j1U3xJk0Jv00jofvwkSYKJNgUpH0ienwATjaR/EbIkyMSWXyrjt8go32ifQNcSbBpCHp
92mGj8LRfp/mhhQmxqQyaUzeZ3r4UB3tN4RuiDMJJP0+zfChONpvWdwQY+JMKpPGpL8YhoNNv610Q5RJIOn3aep4
kZ+ZKBNj4kwqk8ak1x2OR/3O0w0JIv14RkNNEGi4hX7+Rhf/z6VnKASFoagoGomKx/L5tRgKPJbP781Q4NF+LhVD
ESiwWMMeDY+24Z423NPGe0rn9vP+/7EoKASFowgU/EruezqawXwGx6FwFEHiM44MRSPRcBsNj7bh0X4u/m1UPaNQ
FPxTqEe/QTF6RFa/PTEEQkAJOIFKoBHoN4hHj776fNZpLARFoKgoevAyEorCUDiKRqIHG0z8+q2iIQgCn5enjkRG
UVD0E2sjoSgMhaMIFBVFI9Hvmo2ejPX5WM5YOIpA0VD0czt6UHK/XzYWgkJRVBL9XtnoyVifD8CMRUEhKBxFP9rR
iNBvx41FoKgoGon3HbIyeixW/3zNjSgoBIWicBSVhOPROh6t47G8b1fdiEYisGngfgTuR2DT93xxLCo2rdi04tFW
PNqKx1KxuvVio4mDBYpGwgsKReEocE+9kogJRUaBxxK8H9i04k+peG4rFqu4p60XG03oWkZRUAgKQ+Eg+kc3boSh
qCgaiTyhyCgURaCo2LT/lNFcuwWKiqKB6J/LuREZhaBQFI6CevQPjdwI3NNsKHBPM1Uv7y8kKaOHCwqKxtvop3b0
1TGFgBBwAv3FM1hUiBIwAkGgEmgESk8ZI5FRCApHESgain7W60goCkNRSXxenm0kMgpBYSgcRaCoKN7VRw8N7B9h
uhGNxHvdW0aPA+wfn7kRgsJRVBQNxfvsjx4E2D/EcyMMRZDo697RQwD7J2JuhKIwFA1FLzYaEfr6eywcRaCoJPrq
Og8HloyioBAUiqKi6GduNHr0FdhYBIpGoq/ARg/Z6x/WuREFhaAIFBVFQ9HP3Gh86fc0xsJJ9DsWowf09U+r3AhF
YSgcRaCoKBqI/nb7G+EoeD/er9PRowj7p56Gon8C4UZkFPxTFIWhCBSNevR7CaNHHfZPD90IQWEoAkVF0auPRuS+
dh4LR9FA9KnU6BGIfW4xBE4gAPSL7RAogUb70F8Xg7G6TxiHQAgogUqgAehj3xDQYQYdZh9OhsAJBAHs0O9vjR4u
2T8cciMKCkERKCqKhqK/cEZfLtvv1oyFk+hr+9HzIvsnNW6EoXAU/bzUkRAUiiJQVBSNRF/bjx7e2D9ZcSMEhaIw
FIGiomgo3q+g0WMo++dVxqIPSKOnOvaPf9wIReEoGor3uR09FrJ/CuVGGIpAUUn0tb0MB/mMQlAoCkPhKAJFRdFQ
9HM7GqP62n70HMj+UY8bISgMRaBoKPrRjkaPfgdhLBxFJdHnN6PnNvYPeNyIQFFR9B6jK3+fso6FojAUjqKR6Gv7
0WMh++ckboSjeBcbPaqxf3jkRigKQxEoKopGoq9DRw+D7J9ruBGKIlC8z9zoEY79Ixg3wlA4ioqigehD1OjZiv2K
PARCwAg4gUqgEeg3bEaPU7R+O2YsBIWiCBQNRX/52UgYCkdRSfTdGIwrfWIyBAGgD/ZDQPvQB/IhwH2odBR92Tx6
4KT1Je9YOIp+qHUkCgpFYSgCRUXRSPRl8+gxk9YXtGMhKAxFoGgo3mdu9AxK6wvrsXAUlURfNo+eHGl92TwWiiJQ
VBQNRW86Gtj6An8sDIWT6Mvm0SMlrS9Gx6Kh6Ec7GhP6ongsFIWhcBSBopLoc5LRsyatDz1jISgMhaMIFP28jEaP
PkqORUXRSPSRdPRsSOtD6VgUFIJCURgKR1FRNBT93I7GqL5sHj3u0fqCdiwEhaMIFO9zO3pepPVl81gYioqikejL
5tETHK0vm8eioDAUjqKiaCjeZ3/0pEnri/OxCBJ92ezD5VNGYSgcRaBoKPq5HY0e/RbAWCiKCsL7enL0lELvq7Sx
KCgqioai9xgushSFoXAUQaKvo0bPJvS+whkLRdHPSx0JQeEoAkVF0au3kXif/dEz6fqQPXpQXB+xh0AJOIFGoC/2
Rg9/876EGouCQlA4ikDRUPSiMhKGopLoi73RI9+8L/bGQlAoCkdRSRgei+FP6VP1sWi4p/1VOLrY9iXlWASKRsJw
P/qyYywEhaLAo+1Ll6HoC+jRw/O8L37HQlE4il5sNHD0hfxYGIpAUVE0En2xN3pknvfF3lgUFIJCURiKQFFR9LM/
Gkg/Y9RI9MXe6EF53hdZY1FQKApHUVG8z9zoEXneF3tjESgaib7YGz0ez/tibywUhaGoKBqJhsfS8Fj64mYsBAX3
6D9lNHfoy9KxcBRBoi8pR4/v876kHItAUVE0ENEXe6Nn7kVf7I2FoDAUDUU/tz4SisJRBIpKoi/2Rs/+i75QG4uC
QlFUEgX3o68sxgL3o68sxoL3tOHR9j2tI2EoHEWgaCT6mnAsCgo8loLH0tdRo2eqRV/hjEUjYfhT+px/LBQF7qnh
nvY10Ojha9HXlWMhKBSFoQgUlYThsRgei+GeGu6p4Z72ddToUXPR11FjUVAICkXRUPRiZSQcRaCoJPo6avTEuuhr
oLFQFIYiUPRzO/z4SEHhKCqKRqKvo0ZPrIu+jhqLgkJQKApDESgail59NFPq67mh6Ouo0RPros/ox0JRGApHUVH0
sz8akfv6ZSwCRSPR13OjJ9ZFX8+NRUEhKAxFoKgg6lRQCApFYSgcRaBo2ONrT2X0/LV4rzxvRANR36vGG5FRVBRv
MBhM378tY9B3YjAMvq8dY6AEjIATCAKVQCOQe0oZCUVhKBoJwf2QggL3VBxFoKh4tP3FpSNRUAgKRxEoKgnpx2Ij
ISgMRSOh/fXhI1FQCIpAUVH0YjESisJQOIpGwnrTOhIFhaIIFBVFQ9Grj0ZIExSGwkm8V08yep5Ufa9abkRBISgU
haOoKBqKd9PRM6nqe503FtGbDgfKjMJRVBQNRT9zo7EhBIWiMBRBok+hRs+Tqn2KNBaCwlAEit50NDb0qdhYOIqK
opFovelobGgFhaBQFBVFQ9HP/mj0aIbCUQSIT9LBhf+znyNQCAgBJeAE+qpo9HymNgkKQ+EoAkVD8S4+enJSe6+r
b4SiqCT6ymX01KPWVy5jYSgCRUPRi5WRKCgEhaOoJEpvKiORURQU/VhGz+EtisJQOIpAUVE0En1lO3omT+sr27FQ
FIYiUDQU/ez7SAgKR1FJ9NXg6Mk/ra8Gx0JQKApHESgqioain5fhGGYkOhhd+INAJdAI9BXp6Ik9ra/zxqKgUBSG
oqJ4v75Gz/RpfTU5FoGikegr0tETe1pfb46FoFAUhiJQ9LM/Glv6qnYsKopGoq9IR0/saX0lOBaKwlA4ioaiNx1d
9fuadSwCRSXRV6SjJ/a0viIdC0FhKBxFRdHP/uiK3desYxEoGom+fBo9saf19dNYCApFYSh6seHg4SgCRUXRPuJ/
/D9Byv7UXu4BAA=="""
MANIFESTO = json.loads(gzip.decompress(base64.b64decode(DADOS_COMPACTADOS)))
base = globals().get("DATASET_ORIGINAL")
if base is None or not Path(base).is_dir():
    base = kagglehub.dataset_download("chzili/dataset-for-tobacco-leaf-disease-segmentation/versions/1")
BASE = Path(base)
SAIDA = Path("/content/SafraCerta_kaggle_agrupado")
SAIDA.mkdir(exist_ok=True)
pastas = {}
for conjunto in ("data1", "data2"):
    encontradas = [p for p in (BASE / conjunto).rglob("imgs") if p.is_dir()]
    assert len(encontradas) == 1, f"Pasta imgs inesperada: {conjunto}"
    pastas[conjunto] = encontradas[0]

# As relações visuais foram calculadas previamente nas 3.500 miniaturas.
# Também unimos duplicatas exatas identificadas nos pixels originais.
parent = list(range(1 + max(r[2] for r in MANIFESTO)))
def raiz(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def unir(i, j):
    parent[raiz(i)] = raiz(j)

# Cache local: se esta célula for interrompida, reaproveita o que terminou.
# O cache permanece apenas enquanto o ambiente do Colab conservar os arquivos.
CACHE = SAIDA / "preparo_cache_v2"
CACHE.mkdir(exist_ok=True)
registros, hashes = [], {}
inicio = time.monotonic()
reutilizados = 0
print(f"Dataset localizado. Conferindo {len(MANIFESTO)} imagens...", flush=True)
for n, (conjunto, nome, grupo) in enumerate(MANIFESTO, 1):
    foto = pastas[conjunto] / nome
    mascara = pastas[conjunto].parent / "masks" / nome
    chave = hashlib.sha256((conjunto + "/" + nome).encode()).hexdigest()
    meta_path = CACHE / (chave + ".json")
    label_path = CACHE / (chave + ".txt")
    estado = [[str(f.resolve()), f.stat().st_size, f.stat().st_mtime_ns] for f in (foto, mascara)]
    info = None
    if meta_path.is_file() and label_path.is_file():
        try:
            candidato = json.loads(meta_path.read_text())
            if candidato["estado"] == estado and candidato["grupo_original"] == grupo:
                info = candidato
                reutilizados += 1
        except (ValueError, KeyError):
            pass
    if info is None:
        with Image.open(foto) as im:
            pixels = np.asarray(im.convert("RGB"))
        h, w = pixels.shape[:2]
        assinatura = hashlib.sha256(pixels.tobytes()).hexdigest()
        del pixels
        with Image.open(mascara) as im:
            assert im.mode in ("P", "L"), f"Modo inesperado: {mascara}"
            mask = np.array(im)
        assert mask.shape == (h, w), f"Dimensões incompatíveis: {nome}"
        assert set(np.unique(mask)).issubset({0, 1}), f"Valores inesperados: {nome}"
        _, componentes, stats, centros = cv2.connectedComponentsWithStats(
            (mask == 1).astype("uint8"), connectivity=8)
        del componentes, centros, mask
        linhas, pequenas = [], 0
        for x, y, largura, altura, area in stats[1:]:
            linhas.append(f"0 {(x+largura/2)/w:.8f} {(y+altura/2)/h:.8f} {largura/w:.8f} {altura/h:.8f}")
            pequenas += int(largura < 3 or altura < 3)
        label_path.write_text("\n".join(linhas) + ("\n" if linhas else ""))
        info = dict(estado=estado, grupo_original=grupo, sha256_pixels=assinatura,
                    caixas=len(linhas), pequenas=pequenas)
        temporario = meta_path.with_suffix(".tmp")
        temporario.write_text(json.dumps(info))
        temporario.replace(meta_path)
        del linhas, stats
    assinatura = info["sha256_pixels"]
    if assinatura in hashes:
        unir(grupo, hashes[assinatura])
    else:
        hashes[assinatura] = grupo
    registros.append(dict(conjunto=conjunto, nome=nome, grupo=grupo, imagem=str(foto),
                          sha256_pixels=assinatura, caixas=info["caixas"],
                          pequenas=info["pequenas"], label_cache=str(label_path)))
    if n == 1 or n % 50 == 0 or n == len(MANIFESTO):
        print(f"Preparando: {n}/{len(MANIFESTO)} | "
              f"{time.monotonic()-inicio:.0f}s | cache: {reutilizados}", flush=True)

grupos = defaultdict(list)
for r in registros:
    r["grupo"] = raiz(r["grupo"])
    grupos[r["grupo"]].append(r)

# Divisão única e reprodutível. Grupos relacionados nunca são separados.
rng = random.Random(42)
ids = list(grupos)
rng.shuffle(ids)
ids.sort(key=lambda g: len(grupos[g]), reverse=True)
alvos = {"train": .70*len(registros), "val": .15*len(registros), "test": .15*len(registros)}
contagem = Counter()
for g in ids:
    parte = max(alvos, key=lambda s: alvos[s] - contagem[s])
    for r in grupos[g]:
        r["split"] = parte
    contagem[parte] += len(grupos[g])

for parte in alvos:
    (SAIDA / "images" / parte).mkdir(parents=True, exist_ok=True)
    (SAIDA / "labels" / parte).mkdir(parents=True, exist_ok=True)

relatorio = []
for r in registros:
    nome = r["conjunto"] + "__" + r["nome"]
    destino = SAIDA / "images" / r["split"] / nome
    if destino.is_symlink() and destino.resolve() != Path(r["imagem"]).resolve():
        destino.unlink()
    if not destino.exists():
        destino.symlink_to(Path(r["imagem"]).resolve())
    etiqueta = SAIDA / "labels" / r["split"] / (Path(nome).stem + ".txt")
    shutil.copyfile(r["label_cache"], etiqueta)
    relatorio.append({k: v for k, v in r.items() if k != "label_cache"})

for chave in ("grupo", "sha256_pixels"):
    conjuntos = [{r[chave] for r in registros if r["split"] == s} for s in alvos]
    assert all(not conjuntos[i] & conjuntos[j] for i in range(3) for j in range(i+1, 3))

DATA_YAML = SAIDA / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({"path": str(SAIDA), "train": "images/train",
    "val": "images/val", "test": "images/test", "names": {0: "lesao"}}, allow_unicode=True))
(SAIDA / "divisao.json").write_text(json.dumps(relatorio, ensure_ascii=False, indent=2))
print("\nPreparação concluída:", dict(contagem))
print("Grupos de relação visual/nome:", len(grupos))
print("Caixas:", sum(r["caixas"] for r in relatorio))
print("Caixas com lado menor que 3 pixels (preservadas):", sum(r["pequenas"] for r in relatorio))
print("Os grupos reduzem sobreposição conhecida; não comprovam independência entre folhas.")


Iniciando a preparação...


100%|██████████| 2.56G/2.56G [00:24<00:00, 111MB/s]

Extracting files...


Dataset localizado. Conferindo 3500 imagens...
Preparando: 1/3500 | 0s | cache: 0
Preparando: 50/3500 | 3s | cache: 0
Preparando: 100/3500 | 5s | cache: 0
Preparando: 150/3500 | 7s | cache: 0
Preparando: 200/3500 | 10s | cache: 0
Preparando: 250/3500 | 12s | cache: 0
Preparando: 300/3500 | 14s | cache: 0
Preparando: 350/3500 | 16s | cache: 0
Preparando: 400/3500 | 18s | cache: 0
Preparando: 450/3500 | 20s | cache: 0
Preparando: 500/3500 | 23s | cache: 0
Preparando: 550/3500 | 25s | cache: 0
Preparando: 600/3500 | 27s | cache: 0
Preparando: 650/3500 | 30s | cache: 0
Preparando: 700/3500 | 32s | cache: 0
Preparando: 750/3500 | 34s | cache: 0
Preparando: 800/3500 | 36s | cache: 0
Preparando: 850/3500 | 38s | cache: 0
Preparando: 900/3500 | 40s | cache: 0
Preparando: 950/3500 | 42s | cache: 0
Preparando: 1000/3500 | 44s | cache: 0
Preparando: 1050/3500 | 46s | cache: 0
Preparando: 1100/3500 | 49s | cache: 0
Preparando: 1150/3500 | 51s | cache: 0
Preparando: 1200/3500 | 53s | cache: 0
Prepa

In [3]:
import torch
from datetime import datetime
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)
if DEVICE == "cpu":
    print("Com 3.500 imagens, o treinamento em CPU pode levar várias horas.")

# Novo experimento; mantém o V1 e seus resultados intactos.
modelo = YOLO("yolo26n.pt")
resultado_treino = modelo.train(
    data=str(DATA_YAML), epochs=50, patience=12,
    imgsz=800, batch=4, nbs=16, device=DEVICE,
    workers=2 if DEVICE != "cpu" else 0,
    optimizer="AdamW", lr0=0.001, lrf=0.1,
    warmup_epochs=3, weight_decay=0.0005,
    seed=42, deterministic=True,
    mosaic=0.0, mixup=0.0, copy_paste=0.0,
    degrees=5, translate=0.03, scale=0.10,
    fliplr=0.5, flipud=0.0,
    hsv_h=0.01, hsv_s=0.2, hsv_v=0.2,
    cache=False, plots=True, save=True,
    project="/content/SafraCerta_kaggle_runs",
    name="lesoes_" + datetime.now().strftime("%Y%m%d_%H%M%S"),
)
PASTA_TREINO = Path(modelo.trainer.save_dir)
BEST = PASTA_TREINO / "weights" / "best.pt"
assert BEST.is_file()
print("Modelo salvo:", BEST)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Dispositivo: 0
New https://pypi.org/project/ultralytics/8.4.208 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/SafraCerta_kaggle_agrupado/data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=

In [4]:
import shutil, platform
import ultralytics
from google.colab import files

# Apenas o teste reservado deste dataset. Não ajustamos parâmetros por ele.
detector = YOLO(str(BEST))
metricas = detector.val(data=str(DATA_YAML), split="test", imgsz=800,
    batch=4, device=DEVICE, workers=0, plots=True,
    project=str(PASTA_TREINO), name="teste_reservado")
print("Métricas no teste reservado:")
print(json.dumps({k: float(v) for k, v in metricas.results_dict.items()}, indent=2))

ENTREGA = PASTA_TREINO / "entrega"
ENTREGA.mkdir(exist_ok=True)
shutil.copy2(BEST, ENTREGA / "best_lesoes_kaggle.pt")
shutil.copy2(SAIDA / "divisao.json", ENTREGA / "divisao.json")
shutil.copy2(DATA_YAML, ENTREGA / "data.yaml")
for nome in ("results.csv", "args.yaml", "results.png", "confusion_matrix.png"):
    origem = PASTA_TREINO / nome
    if origem.exists(): shutil.copy2(origem, ENTREGA / nome)
if Path(metricas.save_dir).is_dir():
    shutil.copytree(metricas.save_dir, ENTREGA / "teste_reservado", dirs_exist_ok=True)
(ENTREGA / "metricas_teste.json").write_text(json.dumps(
    {k: float(v) for k, v in metricas.results_dict.items()}, indent=2))
(ENTREGA / "ambiente.json").write_text(json.dumps({"python": platform.python_version(),
    "torch": torch.__version__, "ultralytics": ultralytics.__version__, "device": str(DEVICE)}, indent=2))
(ENTREGA / "LEIA-ME.txt").write_text(
    "Experimento de localização de lesões; classe única lesao. Não identifica a doença.\n"
    "Fonte: chzili/dataset-for-tobacco-leaf-disease-segmentation, versão 1.\n"
    "3.500 pares; caixas derivadas dos componentes conectados das máscaras originais.\n"
    "Grupos por captura, família de nomes e correspondências geométricas em miniaturas.\n"
    "Duplicatas exatas dos pixels originais foram unidas antes da divisão.\n"
    "Não há comprovação de independência entre folhas. Pode haver relações não detectadas.\n"
    "As máscaras completas não passaram por validação de especialista nesta revisão.\n"
    "Treino/val/teste por grupo: aproximadamente 70/15/15. Teste interno, não externo.\n"
    "Não compare diretamente estas métricas com as quatro imagens históricas do V1.\n"
    "Os arquivos originais estão no Kaggle; este pacote não inclui o dataset.\n"
    "Ainda é necessário avaliar em fotos novas de campo e definir a confiança na validação.\n",
    encoding="utf-8")
zip_final = shutil.make_archive(str(PASTA_TREINO / "SafraCerta_resultado_kaggle"), "zip", ENTREGA)
files.download(zip_final)


Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 121 layers, 2,389,735 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1783.3±1644.1 MB/s, size: 923.9 KB)
val: Scanning /content/SafraCerta_kaggle_agrupado/labels/test... 525 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 525/525 331.6it/s 1.6s
val: New cache created: /content/SafraCerta_kaggle_agrupado/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 132/132 7.8it/s 17.0s
                   all        525       2234      0.837      0.702      0.767      0.511
Speed: 0.6ms preprocess, 9.2ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/SafraCerta_kaggle_runs/lesoes_20261110_112509/teste_reservado
Métricas no teste reservado:
{
  "metrics/precision(B)": 0.8365306146452041,
  "metrics/recall(B)": 0.7018800358102059,
  "metrics/mAP50

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>